# Notebook 01 · Preparing land-cover observations for indicator analysis

**Exercise:** Carbon storage change in mainland Portugal, 1990–2018
**Companion to:** Pedro Cabral, *Monitoring Dynamics in Ecosystem Service Indicators with Remote Sensing* (Elsevier)

This notebook prepares the CORINE Land Cover (CLC) maps for 1990 and 2018 that Notebook 02 turns into a carbon storage indicator. It follows step 1 of the workflow used in the ESP Summer School tutorial (Barcelona, 2026), and makes each preparation decision explicit, because each one can change the indicator.

**You will learn to**
1. place a land-cover product in the chain from satellite observation to ecosystem service indicator;
2. clip categorical rasters without resampling them, and see why resampling would corrupt them;
3. check that raster class codes match the model's parameter table before running anything;
4. describe land-cover change with area tables and a transition matrix;
5. record what you did so that someone else can repeat it.

| Step | What you do | Book section |
|---|---|---|
| 0 | Frame the chain observation → indicator → service | 1.3, 2.2 (Fig. 2.1) |
| 1 | Obtain and clip CLC 1990 and 2018 | 3.2, 3.3 |
| 2 | Check grids and see why categorical data need nearest neighbour | 3.3 |
| 3 | Check class codes against the carbon table | 4.6.3 |
| 4 | Describe land-cover composition and transitions | 4.2, 5.3 |
| 5 | Record a reproducibility log | 3.6 |

## Step 0 · Where land cover sits in the indicator chain

> **Book link · Sections 1.3 and 2.2.** Remote sensing observes ecosystem *properties*, not services. Indicators are the bridge: a documented transformation of observations into a quantity that is expected to track a service.

In this exercise the chain is:

| Level | In this exercise | Who produced it |
|---|---|---|
| Observation | Satellite imagery (mainly Landsat for the 1990 edition; Sentinel-2 and Landsat 8 for 2018) | Satellite missions |
| Derived product | CLC land-cover classes, 44 classes, minimum mapping unit 25 ha | Photo-interpretation by national teams, coordinated by the EEA |
| Indicator | Carbon storage (t C ha⁻¹), assigned per class by the InVEST carbon model | You, in Notebook 02 |
| Service | Climate regulation through carbon storage | Inferred, not observed |

Two consequences follow, and the notebooks return to both:
- **Every error in CLC passes into the indicator.** A misclassified pixel gets the wrong carbon value.
- **Within-class variation is invisible.** A degraded and a healthy forest share one carbon value as long as they share one CLC class. Notebook 03 uses a satellite time series to look inside the classes.

In [ ]:
import json
import subprocess
import sys
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject

import es_utils as es

DATA = es.DATA_DIR
OUT = es.OUT_DIR
OUT.mkdir(exist_ok=True)
CLC90 = DATA / "clc1990_portugal.tif"
CLC18 = DATA / "clc2018_portugal.tif"
TABLE = DATA / "carbon_pools.csv"

## Step 1 · Obtain and clip CLC

> **Book link · Section 3.2 (data sources) and Section 3.3 (harmonization).**

The full CLC rasters cover Europe at 100 m and are about 150–200 MB each, so the repository ships them already clipped to mainland Portugal. The clipping script `scripts/clip_clc.py` makes one decision worth noticing: **it reprojects the boundary vector to the raster's CRS (EPSG:3035), never the raster to the boundary's CRS.** The CLC pixels therefore keep their original codes, size and alignment.

If the clipped files are missing, download the two rasters from the [Copernicus Land Monitoring Service](https://land.copernicus.eu/en/products/corine-land-cover) (raster, 100 m, version *V2020_20u1*), set the paths below and run the cell.

In [ ]:
RAW_CLC1990 = None  # e.g. Path("D:/data/U2000_CLC1990_V2020_20u1.tif")
RAW_CLC2018 = None  # e.g. Path("D:/data/U2018_CLC2018_V2020_20u1.tif")
BOUNDARY = None     # e.g. Path("D:/data/Portugal.shp")

if CLC90.exists() and CLC18.exists():
    print("Clipped CLC rasters found:")
    for f in (CLC90, CLC18):
        print(f"  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")
elif RAW_CLC1990 and RAW_CLC2018 and BOUNDARY:
    repo_root = Path.cwd().parents[1]
    subprocess.run([sys.executable, str(repo_root / "scripts/clip_clc.py"),
                    "--clc1990", str(RAW_CLC1990), "--clc2018", str(RAW_CLC2018),
                    "--boundary", str(BOUNDARY), "--out", str(DATA)], check=True)
else:
    raise FileNotFoundError("Clipped rasters not found. Set RAW_CLC1990, RAW_CLC2018 and BOUNDARY above.")

## Step 2 · Check that both years share one grid

> **Book link · Section 3.3.** Comparing two maps pixel by pixel is only meaningful if the pixels are the same pixels: same CRS, same size, same alignment. The tutorial slides put it as *"preserve the CLC original CRS, spatial resolution and raster alignment."*

In [ ]:
lc90, prof = es.read_clc(CLC90)
lc18, prof18 = es.read_clc(CLC18)

grid = pd.DataFrame({
    "1990": [str(prof["crs"]), prof["transform"].a, (prof["height"], prof["width"]), prof["nodata"]],
    "2018": [str(prof18["crs"]), prof18["transform"].a, (prof18["height"], prof18["width"]), prof18["nodata"]],
}, index=["CRS", "pixel size (m)", "shape (rows, cols)", "nodata value"])
display(grid)

assert prof["crs"] == prof18["crs"] and prof["transform"] == prof18["transform"] \
    and lc90.shape == lc18.shape, "The two rasters are not on the same grid."
AREA_HA = es.pixel_area_ha(prof)
print(f"Same grid confirmed. One pixel = {AREA_HA:g} ha.")

### Why categorical rasters need nearest-neighbour resampling

> **Book link · Section 3.3.** Bilinear or cubic resampling averages neighbouring values. For reflectance or NDVI that is reasonable; for class codes it is meaningless, because the average of *Broad-leaved forest* (23) and *Mixed forest* (25) is not *Coniferous forest* (24).

The cell below resamples a small window of the 2018 map from 100 m to 150 m in two ways and counts the damage. It is the same operation a GIS performs silently when a categorical layer is reprojected with the wrong setting.

In [ ]:
r0, c0, n = lc18.shape[0] // 2, lc18.shape[1] // 2, 400
window = lc18[r0:r0 + n, c0:c0 + n].astype(np.float32)
window[window < 0] = np.nan
src_t = prof["transform"] * rasterio.Affine.translation(c0, r0)
dst_t = src_t * rasterio.Affine.scale(1.5)
m = int(n / 1.5)

results = {}
for method in (Resampling.nearest, Resampling.bilinear):
    dst = np.full((m, m), np.nan, np.float32)
    reproject(window, dst, src_transform=src_t, dst_transform=dst_t,
              src_crs=prof["crs"], dst_crs=prof["crs"], resampling=method,
              src_nodata=np.nan, dst_nodata=np.nan)
    results[method.name] = dst

present = set(np.unique(window[~np.isnan(window)]).astype(int))
for name, arr in results.items():
    vals = arr[~np.isnan(arr)]
    not_integer = np.mean(vals != np.round(vals)) * 100
    invented = np.mean(~np.isin(np.round(vals).astype(int), list(present))) * 100
    print(f"{name:>8}: {not_integer:5.1f}% non-integer codes, "
          f"{invented:5.1f}% pixels with a class absent from the source window")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (name, arr) in zip(axes, results.items()):
    ax.imshow(arr, cmap="tab20", interpolation="nearest")
    ax.set_title(f"Resampled with {name}")
    ax.axis("off")
plt.tight_layout()

Nearest neighbour keeps every pixel a real class. Bilinear produces fractional codes and, once rounded, classes that did not exist in the source. Each of those pixels would receive a carbon value in Notebook 02 for a land cover that is not there.

Notebook 03 aggregates CLC from 100 m to the 250 m MODIS grid. There it uses the *mode* (majority class), the categorical counterpart of averaging.

## Step 3 · Check class codes against the carbon table

> **Book link · Section 4.6.3 (key input data and their implications).** InVEST assigns carbon by matching each raster value to the `lucode` column of the parameter table. A raster value with no matching row is a silent failure.

A common trap: the CLC raster stores *grid codes* 1–44 (plus 48 for NODATA), not the three-digit CLC nomenclature codes (111, 112, …). The table must use the same scheme as the raster.

In [ ]:
table = pd.read_csv(TABLE)
table["c_total"] = table[es.POOLS].sum(axis=1)

codes_90 = set(np.unique(lc90[lc90 >= 0]))
codes_18 = set(np.unique(lc18[lc18 >= 0]))
table_codes = set(table["lucode"])
print("Codes in the rasters but missing from the table:", sorted((codes_90 | codes_18) - table_codes) or "none")
print("Table rows never used by either raster:", sorted(table_codes - codes_90 - codes_18))

area = pd.DataFrame({
    "1990_ha": pd.Series(np.bincount(lc90[lc90 >= 0], minlength=256) * AREA_HA),
    "2018_ha": pd.Series(np.bincount(lc18[lc18 >= 0], minlength=256) * AREA_HA),
}).loc[sorted(codes_90 | codes_18)]
area = table.set_index("lucode")[["lulc_name", "c_total"]].join(area, how="right")
area["change_ha"] = area["2018_ha"] - area["1990_ha"]
display(area.round(0).sort_values("2018_ha", ascending=False))

## Step 4 · Describe land-cover composition and change

> **Book link · Section 4.2 (what makes a good indicator) and Section 5.3 (degradation signals).** Before computing any indicator, look at what changed in the observations themselves. Every carbon change in Notebook 02 will be produced by one of these transitions.

**A caution on comparing CLC editions (Section 3.2).** The 1990 and 2018 maps were produced from different satellites, at different times, by different interpretation teams. The *V2020_20u1* release revises earlier editions for consistency, but part of any difference between the two maps may still be mapping difference rather than real change. CLC also ignores changes smaller than its minimum mapping unit.

In [ ]:
g90, g18 = es.to_groups(lc90), es.to_groups(lc18)

fig, axes = plt.subplots(1, 2, figsize=(11, 8))
es.plot_groups(axes[0], g90, prof, "CORINE Land Cover 1990", legend=False)
es.plot_groups(axes[1], g18, prof, "CORINE Land Cover 2018")
plt.tight_layout()
fig.savefig(OUT / "01_landcover_1990_2018.png", dpi=150)

In [ ]:
comp = pd.DataFrame({"1990": es.area_by_group(g90, AREA_HA), "2018": es.area_by_group(g18, AREA_HA)})
comp["change_ha"] = comp["2018"] - comp["1990"]
comp["change_%"] = 100 * comp["change_ha"] / comp["1990"].replace(0, np.nan)
comp.to_csv(OUT / "01_landcover_composition.csv")
display((comp / [1e3, 1e3, 1e3, 1]).round(1).rename(columns={"1990": "1990 (kha)", "2018": "2018 (kha)", "change_ha": "change (kha)"}))

In [ ]:
tm = es.transition_matrix(g90, g18, AREA_HA)
tm.to_csv(OUT / "01_transition_matrix_ha.csv")
changed = (lc90 != lc18) & (lc90 >= 0) & (lc18 >= 0)
valid = (lc90 >= 0) & (lc18 >= 0)
print(f"Area whose CLC class changed 1990 to 2018: {changed.sum() * AREA_HA / 1e3:,.0f} kha "
      f"({100 * changed.sum() / valid.sum():.1f}% of the mainland)")

off = tm.where(~np.eye(len(tm), dtype=bool))
top = off.stack().sort_values(ascending=False).head(10) / 1e3
display(top.rename("kha").round(1).to_frame().rename_axis(["from (1990)", "to (2018)"]))

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(np.log10(off.to_numpy() + 1), cmap="Oranges")
ax.set_xticks(range(len(tm)), tm.columns, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(len(tm)), tm.index, fontsize=8)
ax.set_xlabel("2018")
ax.set_ylabel("1990")
ax.set_title("Land-cover transitions (log10 ha, persistence on the diagonal omitted)")
plt.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()

**Questions to ask of these tables before moving on**
- Which three transitions cover most of the changed area? Do they match what you know about Portugal since 1990 (afforestation with eucalyptus and pine, wildfires, abandonment of farmland, coastal urban growth)?
- Is any transition implausible, for example forest becoming arable land and then being mapped as forest again? Such cases often point to mapping differences between editions rather than real change.
- *Open & burnt* in 2018: how much of it came from forest and shrubland? Keep the number in mind for Notebooks 02 and 03.

## Step 5 · Record what was done

> **Book link · Section 3.6 (platform choices and reproducibility).** An indicator that cannot be regenerated cannot be audited. The record below stores the software versions, the input checksums and the grid, so that a later run can confirm it starts from the same inputs.

In [ ]:
import hashlib

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

record = {
    "notebook": "01_data_preparation",
    "inputs": {p.name: sha256(p) for p in (CLC90, CLC18, TABLE)},
    "grid": {"crs": str(prof["crs"]), "pixel_size_m": prof["transform"].a,
             "shape": [prof["height"], prof["width"]]},
    "software": {pkg: version(pkg) for pkg in ("numpy", "pandas", "rasterio", "geopandas", "matplotlib")},
    "python": sys.version.split()[0],
}
(OUT / "01_run_record.json").write_text(json.dumps(record, indent=2))
print(json.dumps(record, indent=2))

## Discussion questions

1. CLC has a 25 ha minimum mapping unit. Which carbon-relevant changes in Portugal are likely to fall below it, and in which direction would their omission bias Notebook 02's result? *(Sections 3.2, 4.2.1)*
2. The 1990 and 2018 maps come from different sensors and interpretation teams. Propose one check that would help separate real change from mapping difference in the transition matrix. *(Sections 3.2, 3.3)*
3. Suppose a colleague reprojected both rasters to the Portuguese national CRS (EPSG:3763) with bilinear resampling before running InVEST. What would you expect to see in the carbon change map, and how would you detect the problem from the outputs alone? *(Section 3.3)*
4. Which entries of the run record would you need to change if you repeated the exercise with the CLC 2006 and 2018 maps instead? What would remain valid? *(Section 3.6)*

**Next:** Notebook 02 turns these maps into carbon storage, quantifies the change, and asks how confident we can be in it.

---
*Data: CORINE Land Cover, © European Union, Copernicus Land Monitoring Service, European Environment Agency (EEA). Mainland boundary derived from the Carta Administrativa Oficial de Portugal (CAOP), Direção-Geral do Território.*